In [ ]:
from pathlib import Path
import nibabel as nib
import torchio as tio
import numpy as np
import torch
from datasets import EigenvalueDataset, EigenvalueVectorDataset, TensorDataset

from torch.utils.data import Dataset, DataLoader

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
raw_data_root = Path("/nfs2/harmonization/BIDS/")
targets = list(raw_data_root.glob("*/derivatives/sub-*/ses-*/PreQual/TENSOR"))

unique = {}
for path in targets:
    parts = path.parts
    try:
        sub = next(p for p in parts if p.startswith("sub-"))
        ses = next(p for p in parts if p.startswith("ses-"))
        key = (sub, ses)
    except StopIteration:
        continue

    if key not in unique:
        unique[key] = path

In [9]:
final_paths = []
for p in unique.values():
    tensor_path = p / "dwmri_tensor.nii.gz"
    try:
        if tensor_path.is_file():
            final_paths.append(tensor_path)
    except PermissionError:
        print(f"[Warning] Permission denied, skipping {tensor_path}")
        continue

print(f"Valid tensor files: {len(final_paths)}")

[Warning] Permission denied, skipping /nfs2/harmonization/BIDS/BLSA/derivatives/sub-BLSA5809/ses-110scanner12/PreQual/TENSOR/dwmri_tensor.nii.gz
[Warning] Permission denied, skipping /nfs2/harmonization/BIDS/BLSA/derivatives/sub-BLSA7453/ses-010scanner12/PreQual/TENSOR/dwmri_tensor.nii.gz
[Warning] Permission denied, skipping /nfs2/harmonization/BIDS/BLSA/derivatives/sub-BLSA7511/ses-020scanner12/PreQual/TENSOR/dwmri_tensor.nii.gz
[Warning] Permission denied, skipping /nfs2/harmonization/BIDS/BLSA/derivatives/sub-BLSA5907/ses-120scanner12/PreQual/TENSOR/dwmri_tensor.nii.gz
[Warning] Permission denied, skipping /nfs2/harmonization/BIDS/BLSA/derivatives/sub-BLSA4659/ses-030scanner12/PreQual/TENSOR/dwmri_tensor.nii.gz
[Warning] Permission denied, skipping /nfs2/harmonization/BIDS/BLSA/derivatives/sub-BLSA4908/ses-100scanner12/PreQual/TENSOR/dwmri_tensor.nii.gz
[Warning] Permission denied, skipping /nfs2/harmonization/BIDS/BLSA/derivatives/sub-BLSA1282/ses-070scanner12/PreQual/TENSOR/dwmri

In [10]:
output_file = "tensor_paths_.txt"

with open(output_file, "w") as f:
    for path in final_paths:
        f.write(str(path) + "\n")

print(f"Saved {len(final_paths)} paths to {output_file}")

Saved 23031 paths to tensor_paths_.txt


In [ ]:
input_file = "tensor_paths.txt"

with open(input_file, "r") as f:
    raw_paths = [Path(line.strip()) for line in f if line.strip()]

print(f"Loaded {len(raw_paths)} paths from {input_file}")

In [11]:
for p in raw_paths:
    try:
        img = nib.load(str(p))
        shape = img.header.get_data_shape()
    except Exception as e:
        shape = None
    print(f"{p}: {shape}")

/nfs2/harmonization/BIDS/WRAP/derivatives/sub-wrap0208/ses-baseline/PreQual/TENSOR/dwmri_tensor.nii.gz: (256, 256, 49, 6)
/nfs2/harmonization/BIDS/WRAP/derivatives/sub-wrap0202/ses-year2/PreQual/TENSOR/dwmri_tensor.nii.gz: (256, 256, 60, 6)
/nfs2/harmonization/BIDS/WRAP/derivatives/sub-wrap0202/ses-baseline/PreQual/TENSOR/dwmri_tensor.nii.gz: (256, 256, 49, 6)
/nfs2/harmonization/BIDS/WRAP/derivatives/sub-wrapL0081/ses-year2/PreQual/TENSOR/dwmri_tensor.nii.gz: (256, 256, 58, 6)
/nfs2/harmonization/BIDS/WRAP/derivatives/sub-wrap0101/ses-baseline/PreQual/TENSOR/dwmri_tensor.nii.gz: (256, 256, 49, 6)
/nfs2/harmonization/BIDS/WRAP/derivatives/sub-wrap0101/ses-year2/PreQual/TENSOR/dwmri_tensor.nii.gz: (256, 256, 60, 6)
/nfs2/harmonization/BIDS/WRAP/derivatives/sub-wrap0811/ses-year3/PreQual/TENSOR/dwmri_tensor.nii.gz: (256, 256, 58, 6)
/nfs2/harmonization/BIDS/WRAP/derivatives/sub-wrap0916/ses-year3/PreQual/TENSOR/dwmri_tensor.nii.gz: (256, 256, 58, 6)
/nfs2/harmonization/BIDS/WRAP/derivati

KeyboardInterrupt: 

In [ ]:
valid_paths = [p for p in raw_paths if p.is_file()]
missing = [p for p in raw_paths if not p.is_file()]

print(f"Found {len(valid_paths)} valid files")

with open(input_file, "w") as f:
    for p in valid_paths:
        f.write(str(p) + "\n")

final_paths = valid_paths

In [ ]:
raw_paths[1232]

In [ ]:
subjects = [tio.Subject(dti=tio.ScalarImage(p)) for p in raw_paths[:100] if p.is_file()]
print(f"Found {len(subjects)} volumes.")

In [ ]:
dataset = EigenvalueDataset(path_list=[str(p) for p in raw_paths[:1]], mode="train")

ratios, values = [], []
for v1, v2 in dataset:
    patch = v1
    nz = (patch != 0).float().mean().item()
    ratios.append(nz)
    vals = v1.flatten().cpu().numpy()
    values.extend(vals)


mean_value = float(np.mean(values))
median_ratio = float(np.median(ratios))
std_value = float(np.std(values))

print(f"Mean of all voxel values: {mean_value:.4f}")
print(f"Median non-zero ratio: {median_ratio:.4f}")
print(f"Std of all voxel values: {std_value:.4f}")